# 阶段 2 · 数据清洗与特征工程

时间派生 → 异常标记 → session 切分 → 分层采样 → parquet 中间层落地。

> 代码逻辑统一放在 `src/`，notebook 只做编排与解读。

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import config as C
import db
C.setup_chinese_font()
%matplotlib inline

## 2.1 时间派生

在 DuckDB 的 `clean` 视图里一次派生 `datetime/date/hour/weekday/is_weekend`，
时间戳按**北京时间**解释，`weekday` 与 pandas 口径一致（0=周一）。

In [ ]:
con = db.connect(memory_limit='8GB')
con.execute('SELECT * FROM clean LIMIT 5').fetchdf()

## 2.2 异常流量标记：单用户单日 pv > 500 → 疑似爬虫（标记而不删除）

In [ ]:
import stage2_cleaning as s2
sus = s2.build_suspect_users(con)
print('疑似爬虫用户数:', len(sus))
sus.head()

## 2.3 Session 切分口径

间隔 > 30 分钟切分新会话。查看切分 SQL：

In [ ]:
print(db.sessionized_sql('events', 30))

## 2.5 中间层落地：按 date 分区的 parquet（后续不再重扫 3.6 GB）

In [ ]:
s2.build_events(con)
db.attach_parquet_layer(con)
s2.build_aggregates(con)
s2.build_user_features(con)

## 2.2' 剔除前后敏感性对比

In [ ]:
sens = s2.bot_sensitivity(con)
sens